*Trabajo Práctica 4*

In [8]:
import cv2
import math
from ultralytics import YOLO

model = YOLO('yolo11n.pt') 
model_mat = YOLO('best.pt')

classNames = ["person", "bicycle", "car", "motorbike", "aeroplane", "bus", "train", "truck", "boat",
              "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat",
              "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
              "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite", "baseball bat",
              "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
              "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange", "broccoli",
              "carrot", "hot dog", "pizza", "donut", "cake", "chair", "sofa", "pottedplant", "bed",
              "diningtable", "toilet", "tvmonitor", "laptop", "mouse", "remote", "keyboard", "cell phone",
              "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
              "teddy bear", "hair drier", "toothbrush"
              ]

vid = cv2.VideoCapture('C0142.MP4')
VEHICLE_CLASSES = [2, 3, 5, 7]  


plate_memory = {} 
MAX_PERSIST_FRAMES = 4 

while True:      
    ret, img = vid.read()
    if not ret:
        break  

    h_img, w_img, _ = img.shape

    results = model.track(img, persist=True, conf=0.09, classes=[0, 1, 2, 3, 5, 7], verbose=False)

    for tid in list(plate_memory.keys()):
        plate_memory[tid]['frames_left'] -= 1
        if plate_memory[tid]['frames_left'] <= 0:
            del plate_memory[tid]

    for r in results:
        boxes = r.boxes

        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls = int(box.cls[0])
            
            track_id = int(box.id[0]) if box.id is not None else None

            color_box = (255, 105, 180) if cls == 3 else (255, 0, 0)
            cv2.rectangle(img, (x1, y1), (x2, y2), color_box, 2)
            lbl = f"#{track_id} " if track_id else ""
            cv2.putText(img, f"{lbl}{classNames[cls]}", (x1, max(y1 - 10, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_box, 2)

            if cls in VEHICLE_CLASSES and (x2 - x1) > 10 and (y2 - y1) > 10:
                bw = x2 - x1
                bh = y2 - y1

                pad_w = int(bw * 0.20)
                pad_h = int(bh * 0.20)

                px1 = max(0, x1 - pad_w)
                py1 = max(0, y1 - pad_h)
                px2 = min(w_img, x2 + pad_w)
                py2 = min(h_img, y2 + pad_h)

                roi = img[py1:py2, px1:px2]
                roi_h, roi_w, _ = roi.shape
                
                conf_val = 0.20 if cls != 3 else 0.12
                r_mat = model_mat(roi, conf=conf_val, imgsz=640, verbose=False)
                
                best_plate = None
                max_score = -1.0

                for mbox in r_mat[0].boxes if r_mat and r_mat[0].boxes is not None else []:
                    mx1, my1, mx2, my2 = map(int, mbox.xyxy[0])
                    mat_conf = float(mbox.conf[0])
                    
                    mw = mx2 - mx1
                    mh = my2 - my1
                    if mh == 0: 
                        continue
                    
                    aspect_ratio = mw / float(mh)

                    is_car_plate = (cls != 3 and 2.1 <= aspect_ratio <= 6.0)
                    is_moto_plate = (cls == 3 and 0.6 <= aspect_ratio <= 2.5)

                    pos_y_centro = (my1 + my2) / 2.0
                    not_in_top_grille = (pos_y_centro > (roi_h * 0.25))

                    if (is_car_plate or is_moto_plate) and not_in_top_grille:
                        score = mat_conf + (pos_y_centro / float(roi_h)) * 0.2
                        
                        if score > max_score:
                            max_score = score
                            best_plate = (px1 + mx1, py1 + my1, px1 + mx2, py1 + my2, math.ceil(mat_conf * 100) / 100)

                if best_plate and track_id is not None:
                    plate_memory[track_id] = {
                        'box': (best_plate[0], best_plate[1], best_plate[2], best_plate[3]),
                        'conf': best_plate[4],
                        'frames_left': MAX_PERSIST_FRAMES
                    }

            if track_id in plate_memory:
                mx1, my1, mx2, my2 = plate_memory[track_id]['box']
                mconf = plate_memory[track_id]['conf']

                cv2.rectangle(img, (mx1, my1), (mx2, my2), (0, 255, 0), 2)
                cv2.putText(img, f"Matricula {mconf}", (mx1, max(my1 - 5, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    cv2.imshow('Vid', img)
    if cv2.waitKey(1) == 27:
        break

vid.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import math
import csv
from ultralytics import YOLO

model = YOLO('yolo11n.pt') 
model_mat = YOLO('best.pt')

classNames = ["person", "bicycle", "car", "motorbike", "aeroplane", "bus", "train", "truck", "boat",
              "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat",
              "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
              "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite", "baseball bat",
              "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
              "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange", "broccoli",
              "carrot", "hot dog", "pizza", "donut", "cake", "chair", "sofa", "pottedplant", "bed",
              "diningtable", "toilet", "tvmonitor", "laptop", "mouse", "remote", "keyboard", "cell phone",
              "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
              "teddy bear", "hair drier", "toothbrush"
              ]

input_video_path = 'C0142.MP4'
vid = cv2.VideoCapture(input_video_path)
VEHICLE_CLASSES = [2, 3, 5, 7] 

fps = int(vid.get(cv2.CAP_PROP_FPS))
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('resultado_procesado.mp4', fourcc, fps, (width, height))

csv_file = open('resultados_deteccion.csv', mode='w', newline='', encoding='utf-8')
csv_writer = csv.writer(csv_file)

csv_writer.writerow([
    'fotograma', 'tipo_objeto', 'confianza', 'identificador_tracking', 
    'x1', 'y1', 'x2', 'y2', 
    'matricula_detectada', 'confianza_matricula', 'mx1', 'my1', 'mx2', 'my2'
])

plate_memory = {}
MAX_PERSIST_FRAMES = 4 

counted_track_ids = set()      
counted_plate_ids = set()      
total_counts = {}         

track_history = {}    
counted_zones = []           

def already_counted_at_location(cx, cy, area, dist_thresh=25):
    """Comprueba si en ese punto exacto (<25px) ya se contó un vehículo de tamaño similar."""
    for (zx, zy, z_area) in counted_zones:
        dist = math.hypot(cx - zx, cy - zy)
        area_diff = abs(area - z_area) / float(z_area + 1e-5)
        if dist < dist_thresh and area_diff < 0.20:
            return True
    return False

frame_count = 0

while True:      
    ret, img = vid.read()
    if not ret:
        break  

    frame_count += 1
    h_img, w_img, _ = img.shape

    results = model.track(img, persist=True, conf=0.15, classes=[0, 1, 2, 3, 5, 7], verbose=False)

    for tid in list(plate_memory.keys()):
        plate_memory[tid]['frames_left'] -= 1
        if plate_memory[tid]['frames_left'] <= 0:
            del plate_memory[tid]

    for r in results:
        boxes = r.boxes

        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            confidence = math.ceil((box.conf[0] * 100)) / 100
            cls = int(box.cls[0])
            track_id = int(box.id[0]) if box.id is not None else None

            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            current_area = (x2 - x1) * (y2 - y1)

            if track_id is not None and track_id not in counted_track_ids:
                if track_id not in track_history:
                    track_history[track_id] = {'start_cx': cx, 'start_cy': cy, 'frames': 1}
                else:
                    track_history[track_id]['frames'] += 1

                if track_history[track_id]['frames'] >= 5:
                    if cls in VEHICLE_CLASSES:
                        desplazamiento = math.hypot(
                            cx - track_history[track_id]['start_cx'],
                            cy - track_history[track_id]['start_cy']
                        )
                        is_moving = desplazamiento >= 16

                        if is_moving or not already_counted_at_location(cx, cy, current_area):
                            counted_track_ids.add(track_id)
                            total_counts[cls] = total_counts.get(cls, 0) + 1
                            if not is_moving:
                                counted_zones.append((cx, cy, current_area))
                    else:
                        counted_track_ids.add(track_id)
                        total_counts[cls] = total_counts.get(cls, 0) + 1

            color_box = (255, 105, 180) if cls == 3 else (255, 0, 0)
            cv2.rectangle(img, (x1, y1), (x2, y2), color_box, 2)
            lbl = f"#{track_id} " if track_id else ""
            cv2.putText(img, f"{lbl}{classNames[cls]}", (x1, max(y1 - 10, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_box, 2)

            if cls in VEHICLE_CLASSES and (x2 - x1) > 10 and (y2 - y1) > 10:
                bw = x2 - x1
                bh = y2 - y1

                pad_w = int(bw * 0.20)
                pad_h = int(bh * 0.20)

                px1 = max(0, x1 - pad_w)
                py1 = max(0, y1 - pad_h)
                px2 = min(w_img, x2 + pad_w)
                py2 = min(h_img, y2 + pad_h)

                roi = img[py1:py2, px1:px2]
                roi_h, _, _ = roi.shape
                
                conf_val = 0.20 if cls != 3 else 0.12
                r_mat = model_mat(roi, conf=conf_val, imgsz=640, verbose=False)
                
                best_plate = None
                max_score = -1.0

                for mbox in r_mat[0].boxes if r_mat and r_mat[0].boxes is not None else []:
                    mx1, my1, mx2, my2 = map(int, mbox.xyxy[0])
                    mat_conf = float(mbox.conf[0])
                    
                    mw = mx2 - mx1
                    mh = my2 - my1
                    if mh == 0: 
                        continue
                    
                    aspect_ratio = mw / float(mh)
                    is_car_plate = (cls != 3 and 2.1 <= aspect_ratio <= 6.0)
                    is_moto_plate = (cls == 3 and 0.6 <= aspect_ratio <= 2.5)

                    pos_y_centro = (my1 + my2) / 2.0
                    not_in_top_grille = (pos_y_centro > (roi_h * 0.25))

                    if (is_car_plate or is_moto_plate) and not_in_top_grille:
                        score = mat_conf + (pos_y_centro / float(roi_h)) * 0.2
                        if score > max_score:
                            max_score = score
                            best_plate = (px1 + mx1, py1 + my1, px1 + mx2, py1 + my2, math.ceil(mat_conf * 100) / 100)

                if best_plate and track_id is not None:
                    plate_memory[track_id] = {
                        'box': (best_plate[0], best_plate[1], best_plate[2], best_plate[3]),
                        'conf': best_plate[4],
                        'frames_left': MAX_PERSIST_FRAMES
                    }
                    counted_plate_ids.add(track_id)

            tiene_matricula = False
            mconf = ""
            mx1, my1, mx2, my2 = "", "", "", ""

            if track_id in plate_memory:
                tiene_matricula = True
                mx1, my1, mx2, my2 = plate_memory[track_id]['box']
                mconf = plate_memory[track_id]['conf']

                cv2.rectangle(img, (mx1, my1), (mx2, my2), (0, 255, 0), 2)
                cv2.putText(img, f"Matricula {mconf}", (mx1, max(my1 - 5, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

            csv_writer.writerow([
                frame_count, classNames[cls], confidence, track_id if track_id is not None else -1, 
                x1, y1, x2, y2, 
                tiene_matricula, mconf, mx1, my1, mx2, my2
            ])

    lines_to_draw = len(total_counts) + 1  
    panel_height = 40 + (lines_to_draw * 22)

    cv2.rectangle(img, (10, 10), (280, panel_height), (0, 0, 0), -1)
    cv2.putText(img, "TOTALES ACUMULADOS:", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

    y_pos = 52
    for cls_id, count in total_counts.items():
        name = classNames[cls_id].upper()
        cv2.putText(img, f"{name}: {count}", (20, y_pos),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
        y_pos += 22

    cv2.putText(img, f"MATRICULAS: {len(counted_plate_ids)}", (20, y_pos),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    
    out.write(img)

    cv2.imshow('Vid', img)
    if cv2.waitKey(1) == 27:
        break

vid.release()
out.release()
csv_file.close()
cv2.destroyAllWindows()

print("\n Resumen del vídeo final")
for cls_id, count in total_counts.items():
    print(f"{classNames[cls_id].capitalize()}: {count}")
print(f"Matrículas detectadas: {len(counted_plate_ids)}")


--- RESUMEN FINAL DEL VIDEO ---
Bus: 1
Car: 38
Person: 16
Bicycle: 1
Motorbike: 1
Matrículas detectadas: 28


In [ ]:
import cv2
import math
from ultralytics import YOLO

model = YOLO('yolo11n.pt') 
model_mat = YOLO('best.pt')

classNames = ["person", "bicycle", "car", "motorbike", "aeroplane", "bus", "train", "truck", "boat",
              "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat",
              "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
              "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite", "baseball bat",
              "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
              "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange", "broccoli",
              "carrot", "hot dog", "pizza", "donut", "cake", "chair", "sofa", "pottedplant", "bed",
              "diningtable", "toilet", "tvmonitor", "laptop", "mouse", "remote", "keyboard", "cell phone",
              "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
              "teddy bear", "hair drier", "toothbrush"]

vid = cv2.VideoCapture('C0142.MP4')
fps = int(vid.get(cv2.CAP_PROP_FPS))
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter('resultado_anonimizado.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

VEHICLE_CLASSES = [2, 3, 5, 7] 
plate_memory = {}
MAX_PERSIST_FRAMES = 4 

track_history = {}
already_counted = set()
counts = {'person': {'alejan': 0, 'acercan': 0}, 'vehicle': {'alejan': 0, 'acercan': 0}}

MIN_FRAMES_EVALUATION = 40  
MIN_DISPLACEMENT = 20   
MIN_AREA_CHANGE_RATIO = 0.25 

def blur_roi(img, x1, y1, x2, y2, factor=35):
    h_img, w_img = img.shape[:2]
    x1_c, y1_c = max(0, min(w_img, int(x1))), max(0, min(h_img, int(y1)))
    x2_c, y2_c = max(0, min(w_img, int(x2))), max(0, min(h_img, int(y2)))
    
    if (x2_c - x1_c) > 5 and (y2_c - y1_c) > 5:
        roi = img[y1_c:y2_c, x1_c:x2_c]
        k_size = factor if factor % 2 != 0 else factor + 1
        img[y1_c:y2_c, x1_c:x2_c] = cv2.GaussianBlur(roi, (k_size, k_size), 30)

while True:      
    ret, img = vid.read()
    if not ret:
        break  

    h_img, w_img, _ = img.shape

    results = model.track(img, persist=True, conf=0.06, classes=[0, 1, 2, 3, 5, 7], verbose=False)

    for tid in list(plate_memory.keys()):
        plate_memory[tid]['frames_left'] -= 1
        if plate_memory[tid]['frames_left'] <= 0:
            del plate_memory[tid]

    for r in results:
        boxes = r.boxes
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls = int(box.cls[0])
            track_id = int(box.id[0]) if box.id is not None else -1
            tipo_objeto = classNames[cls]

            if track_id != -1:
                cy = (y1 + y2) // 2
                current_area = (x2 - x1) * (y2 - y1)

                if track_id not in track_history:
                    track_history[track_id] = {'start_area': current_area, 'start_cy': cy, 'frames': 1}
                else:
                    track_history[track_id]['frames'] += 1

                if track_history[track_id]['frames'] == 5:
                    track_history[track_id]['start_area'] = current_area
                    track_history[track_id]['start_cy'] = cy

                if track_id not in already_counted and track_history[track_id]['frames'] >= MIN_FRAMES_EVALUATION:
                    start_area = track_history[track_id]['start_area']
                    start_cy = track_history[track_id]['start_cy']
                    
                    delta_cy = cy - start_cy
                    area_change_ratio = abs(current_area - start_area) / float(start_area + 1e-5)

                    if cls in VEHICLE_CLASSES:
                        is_moving = (area_change_ratio >= MIN_AREA_CHANGE_RATIO) or (abs(delta_cy) >= MIN_DISPLACEMENT)

                        if is_moving:
                            if current_area < start_area or delta_cy <= -10:
                                counts['vehicle']['alejan'] += 1
                                already_counted.add(track_id)
                            else:
                                counts['vehicle']['acercan'] += 1
                                already_counted.add(track_id)

                    elif cls == 0:
                        if delta_cy <= -10:
                            counts['person']['alejan'] += 1
                            already_counted.add(track_id)
                        elif delta_cy >= 10:
                            counts['person']['acercan'] += 1
                            already_counted.add(track_id)

            if cls in VEHICLE_CLASSES and (x2 - x1) > 10 and (y2 - y1) > 10:
                bw, bh = x2 - x1, y2 - y1
                pad_w, pad_h = int(bw * 0.20), int(bh * 0.20)
                px1, py1 = max(0, x1 - pad_w), max(0, y1 - pad_h)
                px2, py2 = min(w_img, x2 + pad_w), min(h_img, y2 + pad_h)

                roi = img[py1:py2, px1:px2]
                roi_h, _, _ = roi.shape
                
                r_mat = model_mat(roi, conf=0.20 if cls != 3 else 0.12, imgsz=640, verbose=False)
                
                best_plate, max_score = None, -1.0
                for mbox in r_mat[0].boxes if r_mat and r_mat[0].boxes is not None else []:
                    mx1, my1, mx2, my2 = map(int, mbox.xyxy[0])
                    mat_conf = float(mbox.conf[0])
                    mw, mh = mx2 - mx1, my2 - my1
                    if mh == 0: continue
                    
                    aspect_ratio = mw / float(mh)
                    is_car = (cls != 3 and 2.1 <= aspect_ratio <= 6.0)
                    is_moto = (cls == 3 and 0.6 <= aspect_ratio <= 2.5)
                    pos_y_centro = (my1 + my2) / 2.0

                    if (is_car or is_moto) and (pos_y_centro > (roi_h * 0.25)):
                        score = mat_conf + (pos_y_centro / float(roi_h)) * 0.2
                        if score > max_score:
                            max_score = score
                            best_plate = (px1 + mx1, py1 + my1, px1 + mx2, py1 + my2, math.ceil(mat_conf * 100) / 100)

                if best_plate and track_id != -1:
                    plate_memory[track_id] = {
                        'box': (best_plate[0], best_plate[1], best_plate[2], best_plate[3]),
                        'conf': best_plate[4],
                        'frames_left': MAX_PERSIST_FRAMES
                    }

            tiene_mat = track_id in plate_memory
            if cls == 0:
                blur_roi(img, x1, y1, x2, y2, factor=35)
            elif cls in VEHICLE_CLASSES:
                if tiene_mat:
                    mx1, my1, mx2, my2 = plate_memory[track_id]['box']
                    blur_roi(img, mx1, my1, mx2, my2, factor=21)
                else:
                    blur_roi(img, x1, y1, x2, y2, factor=35)

            color_box = (255, 105, 180) if cls == 3 else (255, 0, 0)
            cv2.rectangle(img, (x1, y1), (x2, y2), color_box, 2)
            lbl = f"#{track_id} " if track_id != -1 else ""
            cv2.putText(img, f"{lbl}{tipo_objeto}", (x1, max(y1 - 10, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_box, 2)

    cv2.putText(img, f"Vehiculos -> Alejan: {counts['vehicle']['alejan']} | Acercan: {counts['vehicle']['acercan']}", 
                (30, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 255), 2)
    cv2.putText(img, f"Personas  -> Alejan: {counts['person']['alejan']} | Acercan: {counts['person']['acercan']}", 
                (30, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 255), 2)

    out.write(img)
    cv2.imshow('Vid', img)
    if cv2.waitKey(1) == 27:
        break

print("\n Resultados Finales")
print(f"Vehiculos que se alejan: {counts['vehicle']['alejan']} y vehículos que se acercan: {counts['vehicle']['acercan']}")
print(f"Personas que se alejan: {counts['person']['alejan']} y personas que se acercan: {counts['person']['acercan']}")

vid.release()
out.release()
cv2.destroyAllWindows()


--- RESULTADOS FINALES ---
Vehiculos -> Alejan: 18 | Acercan: 16
Personas  -> Alejan: 10 | Acercan: 2


In [9]:
import cv2
import math
import numpy as np
from ultralytics import YOLO

model = YOLO('yolo11n.pt') 

classNames = ["person", "bicycle", "car", "motorbike", "aeroplane", "bus", "train", "truck", "boat",
              "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat",
              "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
              "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite", "baseball bat",
              "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
              "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange", "broccoli",
              "carrot", "hot dog", "pizza", "donut", "cake", "chair", "sofa", "pottedplant", "bed",
              "diningtable", "toilet", "tvmonitor", "laptop", "mouse", "remote", "keyboard", "cell phone",
              "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
              "teddy bear", "hair drier", "toothbrush"
              ]

vid = cv2.VideoCapture('C0142.MP4')
VEHICLE_CLASSES = [2, 3, 5, 7] 

fps = int(vid.get(cv2.CAP_PROP_FPS))
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('resultado_contornos.mp4', fourcc, fps, (width, height))

plate_memory = {}
MAX_PERSIST_FRAMES = 4

def detect_plate_contours(roi, cls):
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

    blurred = cv2.bilateralFilter(gray, 11, 17, 17)

    edged = cv2.Canny(blurred, 30, 200)

    contours, _ = cv2.findContours(edged.copy(), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:10]
    
    roi_h, roi_w = roi.shape[:2]
    best_candidate = None
    
    for c in contours:
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.018 * peri, True)

        if len(approx) == 4:
            x, y, w, h = cv2.boundingRect(approx)
            if h == 0:
                continue
                
            aspect_ratio = w / float(h)
            area = w * h

            is_car_plate = (cls != 3 and 2.1 <= aspect_ratio <= 6.0)
            is_moto_plate = (cls == 3 and 0.6 <= aspect_ratio <= 2.5)
            
            valid_area = (area > 150) and (area < (roi_h * roi_w * 0.4))
            not_in_top_grille = ((y + h / 2.0) > (roi_h * 0.20))
            
            if (is_car_plate or is_moto_plate) and valid_area and not_in_top_grille:
                best_candidate = (x, y, x + w, y + h, 0.85) 
                break
                
    return best_candidate

while True:      
    ret, img = vid.read()
    if not ret:
        break  

    h_img, w_img, _ = img.shape

    results = model.track(img, persist=True, conf=0.09, classes=[0, 1, 2, 3, 5, 7], verbose=False)

    for tid in list(plate_memory.keys()):
        plate_memory[tid]['frames_left'] -= 1
        if plate_memory[tid]['frames_left'] <= 0:
            del plate_memory[tid]

    for r in results:
        boxes = r.boxes

        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls = int(box.cls[0])
            
            track_id = int(box.id[0]) if box.id is not None else None

            color_box = (255, 105, 180) if cls == 3 else (255, 0, 0)
            cv2.rectangle(img, (x1, y1), (x2, y2), color_box, 2)
            lbl = f"#{track_id} " if track_id else ""
            cv2.putText(img, f"{lbl}{classNames[cls]}", (x1, max(y1 - 10, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_box, 2)

            if cls in VEHICLE_CLASSES and (x2 - x1) > 10 and (y2 - y1) > 10:
                bw = x2 - x1
                bh = y2 - y1

                pad_w = int(bw * 0.20)
                pad_h = int(bh * 0.20)

                px1 = max(0, x1 - pad_w)
                py1 = max(0, y1 - pad_h)
                px2 = min(w_img, x2 + pad_w)
                py2 = min(h_img, y2 + pad_h)

                roi = img[py1:py2, px1:px2]

                plate_res = detect_plate_contours(roi, cls)

                if plate_res and track_id is not None:
                    mx1, my1, mx2, my2, c_conf = plate_res
                    plate_memory[track_id] = {
                        'box': (px1 + mx1, py1 + my1, px1 + mx2, py1 + my2),
                        'conf': c_conf,
                        'frames_left': MAX_PERSIST_FRAMES
                    }

            if track_id in plate_memory:
                mx1, my1, mx2, my2 = plate_memory[track_id]['box']
                mconf = plate_memory[track_id]['conf']

                cv2.rectangle(img, (mx1, my1), (mx2, my2), (0, 255, 0), 2)
                cv2.putText(img, f"Matricula {mconf}", (mx1, max(my1 - 5, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    out.write(img)

    cv2.imshow('Vid', img)
    if cv2.waitKey(1) == 27:
        break

vid.release()
out.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import math
import time
from ultralytics import YOLO, RTDETR

model_general = YOLO('yolo11n.pt') 

model_yolo = YOLO('best.pt')
model_rtdetr = RTDETR('bestRTDER.pt')

input_video_path = 'C0142.MP4'
vid = cv2.VideoCapture(input_video_path)

fps = int(vid.get(cv2.CAP_PROP_FPS))
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter('resultado_comparativa.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

classNames = ["person", "bicycle", "car", "motorbike", "aeroplane", "bus", "train", "truck", "boat",
              "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat",
              "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
              "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite", "baseball bat",
              "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
              "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange", "broccoli",
              "carrot", "hot dog", "pizza", "donut", "cake", "chair", "sofa", "pottedplant", "bed",
              "diningtable", "toilet", "tvmonitor", "laptop", "mouse", "remote", "keyboard", "cell phone",
              "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
              "teddy bear", "hair drier", "toothbrush"
              ]

VEHICLE_CLASSES = [2, 3, 5, 7] 

plate_mem_yolo = {}
plate_mem_rtdetr = {}
MAX_PERSIST_FRAMES = 4

stats = {
    'yolo': {'total_time': 0.0, 'rois': 0, 'detections': set()},
    'rtdetr': {'total_time': 0.0, 'rois': 0, 'detections': set()}
}

def eval_plate_model(model_obj, roi, cls, roi_h):
    conf_val = 0.20 if cls != 3 else 0.12
    t_start = time.time()
    results = model_obj(roi, conf=conf_val, imgsz=640, verbose=False)
    t_inference = (time.time() - t_start) * 1000

    best_plate = None
    max_score = -1.0

    for mbox in results[0].boxes if results and results[0].boxes is not None else []:
        mx1, my1, mx2, my2 = map(int, mbox.xyxy[0])
        mat_conf = float(mbox.conf[0])
        mw, mh = mx2 - mx1, my2 - my1
        if mh == 0: 
            continue
        
        aspect_ratio = mw / float(mh)
        is_car_plate = (cls != 3 and 2.1 <= aspect_ratio <= 6.0)
        is_moto_plate = (cls == 3 and 0.6 <= aspect_ratio <= 2.5)
        pos_y_centro = (my1 + my2) / 2.0
        not_in_top_grille = (pos_y_centro > (roi_h * 0.25))

        if (is_car_plate or is_moto_plate) and not_in_top_grille:
            score = mat_conf + (pos_y_centro / float(roi_h)) * 0.2
            if score > max_score:
                max_score = score
                best_plate = (mx1, my1, mx2, my2, math.ceil(mat_conf * 100) / 100)

    return best_plate, t_inference

while True:      
    ret, img = vid.read()
    if not ret:
        break  

    h_img, w_img, _ = img.shape

    results = model_general.track(img, persist=True, conf=0.09, classes=[0, 1, 2, 3, 5, 7], verbose=False)

    for tid in list(plate_mem_yolo.keys()):
        plate_mem_yolo[tid]['frames_left'] -= 1
        if plate_mem_yolo[tid]['frames_left'] <= 0: del plate_mem_yolo[tid]

    for tid in list(plate_mem_rtdetr.keys()):
        plate_mem_rtdetr[tid]['frames_left'] -= 1
        if plate_mem_rtdetr[tid]['frames_left'] <= 0: del plate_mem_rtdetr[tid]

    for r in results:
        boxes = r.boxes

        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls = int(box.cls[0])
            track_id = int(box.id[0]) if box.id is not None else None
            tipo_objeto = classNames[cls]

            color_box = (255, 105, 180) if cls == 3 else (255, 0, 0)
            cv2.rectangle(img, (x1, y1), (x2, y2), color_box, 2)
            lbl = f"#{track_id} " if track_id else ""
            cv2.putText(img, f"{lbl}{tipo_objeto}", (x1, max(y1 - 10, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_box, 2)

            if cls in VEHICLE_CLASSES and (x2 - x1) > 10 and (y2 - y1) > 10:
                bw, bh = x2 - x1, y2 - y1
                pad_w, pad_h = int(bw * 0.20), int(bh * 0.20)

                px1, py1 = max(0, x1 - pad_w), max(0, y1 - pad_h)
                px2, py2 = min(w_img, x2 + pad_w), min(h_img, y2 + pad_h)

                roi = img[py1:py2, px1:px2]
                roi_h, _, _ = roi.shape

                plate_yolo, t_yolo = eval_plate_model(model_yolo, roi, cls, roi_h)
                stats['yolo']['total_time'] += t_yolo
                stats['yolo']['rois'] += 1

                plate_rtdetr, t_rtdetr = eval_plate_model(model_rtdetr, roi, cls, roi_h)
                stats['rtdetr']['total_time'] += t_rtdetr
                stats['rtdetr']['rois'] += 1

                if plate_yolo and track_id is not None:
                    yolo_conf = plate_yolo[4]
                    stats['yolo']['detections'].add(track_id)
                    plate_mem_yolo[track_id] = {
                        'box': (px1 + plate_yolo[0], py1 + plate_yolo[1], px1 + plate_yolo[2], py1 + plate_yolo[3]),
                        'conf': yolo_conf, 'frames_left': MAX_PERSIST_FRAMES
                    }

                if plate_rtdetr and track_id is not None:
                    rtdetr_conf = plate_rtdetr[4]
                    stats['rtdetr']['detections'].add(track_id)
                    plate_mem_rtdetr[track_id] = {
                        'box': (px1 + plate_rtdetr[0], py1 + plate_rtdetr[1], px1 + plate_rtdetr[2], py1 + plate_rtdetr[3]),
                        'conf': rtdetr_conf, 'frames_left': MAX_PERSIST_FRAMES
                    }

            if track_id in plate_mem_yolo:
                mx1, my1, mx2, my2 = plate_mem_yolo[track_id]['box']
                cv2.rectangle(img, (mx1, my1), (mx2, my2), (0, 255, 0), 2) 
                cv2.putText(img, f"YOLO {plate_mem_yolo[track_id]['conf']}", (mx1, max(my1 - 5, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 2)

            if track_id in plate_mem_rtdetr:
                mx1, my1, mx2, my2 = plate_mem_rtdetr[track_id]['box']
                cv2.rectangle(img, (mx1 + 2, my1 + 2), (mx2 - 2, my2 - 2), (0, 255, 255), 1)
                cv2.putText(img, f"RTDETR {plate_mem_rtdetr[track_id]['conf']}", (mx1, min(my2 + 15, h_img - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 2)

    cv2.rectangle(img, (10, 10), (320, 90), (0, 0, 0), -1)
    cv2.putText(img, f"YOLO (Verde)   : {len(stats['yolo']['detections'])} matr. | " +
                f"{stats['yolo']['total_time'] / max(1, stats['yolo']['rois']):.1f} ms",
                (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    cv2.putText(img, f"RT-DETR (Amar.): {len(stats['rtdetr']['detections'])} matr. | " +
                f"{stats['rtdetr']['total_time'] / max(1, stats['rtdetr']['rois']):.1f} ms",
                (20, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)

    out.write(img)
    cv2.imshow('Vid', img)
    if cv2.waitKey(1) == 27:
        break

vid.release()
out.release()
cv2.destroyAllWindows()

avg_yolo_ms = stats['yolo']['total_time'] / max(1, stats['yolo']['rois'])
avg_rtdetr_ms = stats['rtdetr']['total_time'] / max(1, stats['rtdetr']['rois'])

print("\n" + "="*50)
print("        INFORME COMPARATIVO: YOLO vs RT-DETR        ")
print("="*50)
print(f"ROIs de vehículos analizadas : {stats['yolo']['rois']}")
print("-"*50)
print(f"YOLO (`best.pt`):")
print(f"  - Matrículas únicas detectadas : {len(stats['yolo']['detections'])}")
print(f"  - Tiempo medio por ROI         : {avg_yolo_ms:.2f} ms")
print("-"*50)
print(f"RT-DETR (`bestRTDER.pt`):")
print(f"  - Matrículas únicas detectadas : {len(stats['rtdetr']['detections'])}")
print(f"  - Tiempo medio por ROI         : {avg_rtdetr_ms:.2f} ms")
print("="*50)


        INFORME COMPARATIVO: YOLO vs RT-DETR        
ROIs de vehículos analizadas : 6746
--------------------------------------------------
YOLO (`best.pt`):
  - Matrículas únicas detectadas : 28
  - Tiempo medio por ROI         : 69.79 ms
--------------------------------------------------
RT-DETR (`bestRTDER.pt`):
  - Matrículas únicas detectadas : 29
  - Tiempo medio por ROI         : 813.61 ms
